# Order Reviews table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).
Needs cleaned/orders_clean.csv from the orders notebook for the join check.

Decisions and findings for this table are written up in `cleaning_notes/README_order_reviews.md`.

### 1. Imports and settings

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw reviews file

In [ ]:
order_reviews = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')

### 3. First inspection: structure, nulls, preview, score distribution

In [ ]:
order_reviews.info()
print(order_reviews.isna().sum())       # title 87,656 and message 58,247 empty: normal, not an error
print(order_reviews.head())
print(order_reviews['review_score'].value_counts().sort_index())
# 1: 11,424 / 2: 3,151 / 3: 8,179 / 4: 19,142 / 5: 57,328

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   

### 4. Convert the two date columns from text to datetime

In [ ]:
date_cols = ['review_creation_date', 'review_answer_timestamp']
order_reviews[date_cols] = order_reviews[date_cols].apply(pd.to_datetime)

### 5. Key check: review_id alone is NOT unique, the PAIR (review_id, order_id) is

In [ ]:
print(order_reviews['review_id'].is_unique)                                  # False
print(order_reviews['review_id'].duplicated().sum())                         # 814 repeated review_ids
print(order_reviews['review_id'].nunique())                                  # 98,410
print(order_reviews.duplicated(subset=['review_id', 'order_id']).sum())      # 0: the pair is unique
print(order_reviews['order_id'].nunique())                                   # 98,673 orders have a review
print((order_reviews['order_id'].value_counts() > 1).sum())                  # 547 orders have more than one review

False
814
98410
0
98673
547


### 6. Date sanity: an answer should not come before the survey was created

In [ ]:
print((order_reviews['review_answer_timestamp'] < order_reviews['review_creation_date']).sum())   # 0

0


### 7. Join integrity with orders

In [ ]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['order_id', 'order_status'])

print(order_reviews['order_id'].isin(orders['order_id']).sum())              # 99,224: no orphan reviews
print(orders['order_id'].isin(order_reviews['order_id']).sum())              # 98,673
print(orders.loc[~orders['order_id'].isin(order_reviews['order_id']), 'order_status'].value_counts())
# delivered 646 / shipped 75 / canceled 20 / unavailable 14 / processing 6 / invoiced 5 / created 2

99224
98673
order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64


### 8. Look at orders with more than one review

In [ ]:
vc = order_reviews['order_id'].value_counts()
print(order_reviews[order_reviews['order_id'].isin(vc[vc > 1].index)].sort_values('order_id').head(6))

                              review_id                          order_id  \
25612  89a02c45c340aeeb1354a24e7d4b2c1e  0035246a40f520710769010f752e7507   
22423  2a74b0559eb58fc1ff842ecc999594cb  0035246a40f520710769010f752e7507   
22779  ab30810c29da5da8045216f0f62652a2  013056cfe49763c6f66bda03396c5ee3   
68633  73413b847f63e02bc752b364f6d05ee9  013056cfe49763c6f66bda03396c5ee3   
854    830636803620cdf8b6ffaf1b2f6e92b2  0176a6846bcb3b0d3aa3116a9a768597   
83224  d8e8c42271c8fb67b9dad95d98c8ff80  0176a6846bcb3b0d3aa3116a9a768597   

       review_score review_comment_title  \
25612             5                  NaN   
22423             5                  NaN   
22779             5                  NaN   
68633             4                  NaN   
854               5                  NaN   
83224             5                  NaN   

                                                                                                            review_comment_message  \
25612            

### 9. Engineered columns

In [ ]:
# Latest review per order (newest answer time). Avoids double counting orders with 2+ reviews.
order_reviews['is_latest_review'] = ~order_reviews.sort_values(
    ['order_id', 'review_answer_timestamp']
).duplicated(subset='order_id', keep='last')
print(order_reviews['is_latest_review'].sum())      # 98,673 = number of orders with a review

# Did the customer write a comment? (empty comments stay empty, they are not filled)
order_reviews['has_comment'] = order_reviews['review_comment_message'].notna()
print(order_reviews['has_comment'].sum())           # 40,977

98673
40977


### 10. Save the cleaned table and verify the file on disk

In [ ]:
order_reviews.to_csv(CLEAN / 'order_reviews_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_reviews_clean.csv', parse_dates=date_cols)
print(check.shape)                                                          # expect (99224, 9)
print(check.duplicated(subset=['review_id', 'order_id']).sum())             # expect 0
print(check['is_latest_review'].sum())                                      # 98,673
print(check['has_comment'].sum())                                           # 40,977
print(check['review_score'].value_counts().sort_index())                    # same as before
print(check[date_cols].dtypes)                                              # both datetime64

(99224, 9)
0
98673
40977
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object
